## Getting documents

In [1]:
!uv add requests

Resolved 119 packages in 4ms
Checked 116 packages in 18ms


In [2]:
import requests

In [3]:
repo_owner = 'evidentlyai'
repo_name = 'docs'
branch_name = 'main'

zip_url = f'https://github.com/{repo_owner}/{repo_name}/archive/refs/heads/{branch_name}.zip'
zip_response = requests.get(zip_url)

In [4]:
len(zip_response.content)

17545754

In [5]:
import io
import zipfile

zip_archive = zipfile.ZipFile(io.BytesIO(zip_response.content))

In [6]:
filenames = zip_archive.namelist()
filenames[20:30]

['docs-main/docs/library/prompt_optimization.mdx',
 'docs-main/docs/library/report.mdx',
 'docs-main/docs/library/synthetic_data_api.mdx',
 'docs-main/docs/library/tags_metadata.mdx',
 'docs-main/docs/library/tests.mdx',
 'docs-main/docs/platform/',
 'docs-main/docs/platform/alerts.mdx',
 'docs-main/docs/platform/dashboard_add_panels.mdx',
 'docs-main/docs/platform/dashboard_add_panels_ui.mdx',
 'docs-main/docs/platform/dashboard_overview.mdx']

In [7]:
filename = 'docs-main/docs/platform/alerts.mdx'

mdx_file = zip_archive.open(filename)
mdx_content = mdx_file.read().decode('utf8')

In [8]:
!uv add python-frontmatter

Resolved 119 packages in 5ms
Checked 116 packages in 2ms


In [9]:
import frontmatter

post = frontmatter.loads(mdx_content)

In [10]:
post.metadata

{'title': 'Alerts', 'description': 'How to set up alerts.'}

In [11]:
print(post.content[:100])

<Check>
  Built-in alerting is a Pro feature available in the **Evidently Cloud** and **Evidently En


In [12]:
_, filename_corrected = filename.split('/', maxsplit=1)
print(filename_corrected)

docs/platform/alerts.mdx


In [13]:
doc = {
    'content': post.content,
    'title': post.metadata.get('title'),
    'description': post.metadata.get('description'),
    'filename': filename_corrected
}

In [14]:


documents = []
with zipfile.ZipFile(io.BytesIO(zip_response.content)) as zip_ref:
    for file_path in zip_ref.namelist():
        if not file_path.endswith(('.md', '.mdx')):
            continue
        with zip_ref.open(file_path) as file:
            content = file.read().decode('utf-8')
            post = frontmatter.loads(content)
            doc = {
                'content': post.content,
                'title': post.metadata.get('title'),
                'description': post.metadata.get('description'),
                'filename': file_path.split('/', 1)[-1]
            }
            documents.append(doc)



In [15]:
len(documents)

95

In [16]:
!uv add gitsource

Resolved 119 packages in 4ms
Checked 116 packages in 2ms


In [17]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="evidentlyai",
    repo_name="docs",
    allowed_extensions={"md", "mdx"},
)

files = reader.read()

print(f"Loaded {len(files)} documents")

Loaded 95 documents


In [18]:
md_file = files[10]

In [19]:
documents = [f.parse() for f in files]

In [20]:
len(documents)

95

In [21]:
documents[10]

{'title': 'Output formats',
 'description': 'How to export the evaluation results.',
 'content': 'You can view or export Reports in multiple formats.\n\n**Pre-requisites**:\n\n* You know how to [generate Reports](/docs/library/report).\n\n## Log to Workspace\n\nYou can save the computed Report in Evidently Cloud or your local workspace.\n\n```python\nws.add_run(project.id, my_eval, include_data=False)\n```\n\n<Info>\n  **Uploading evals**. Check Quickstart examples [for ML](/quickstart_ml) or [for LLM](/quickstart_llm) for a full workflow.\n</Info>\n\n## View in Jupyter notebook\n\nYou can directly render the visual summary of evaluation results in interactive Python environments like Jupyter notebook or Colab.\n\nAfter running the Report, simply call the resulting Python object:\n\n```python\nmy_report\n```\n\nThis will render the HTML object directly in the notebook cell.\n\n## HTML\n\nYou can also save this interactive visual Report as an HTML file to open in a browser:\n\n```python

## Search

In [22]:
query = 'LLM as a Judge'

In [23]:
!uv add minsearch

Resolved 119 packages in 4ms
Checked 116 packages in 2ms


In [42]:
from minsearch import Index

In [25]:
index = Index(
    text_fields=["title", "description", "content"],
    keyword_fields=["filename"]
)
index.fit(documents)

In [26]:
results = index.search(query, num_results=5)

In [27]:
len(results)

5

In [28]:
len(results[0]['content'])

21834

## Chunking

In [29]:
doc_sizes = [(doc.filename, len(doc.content)) for doc in files]
doc_sizes.sort(key=lambda x: x[1], reverse=True)

for filename, size in doc_sizes[:5]:
    print(f"{filename}: {size} characters")

metrics/all_metrics.mdx: 55085 characters
metrics/all_descriptors.mdx: 31976 characters
docs/platform/dashboard_panel_types.mdx: 31647 characters
docs/library/leftover_content.mdx: 28742 characters
metrics/customize_llm_judge.mdx: 26847 characters


In [30]:
document = list(range(0, 100))

In [31]:
window_size = 10
start = 0
step = 5

chunks = []

while start < len(document):
    end = start + window_size
    chunk = document[start:end]
    if len(chunk) < window_size:
        break
    chunks.append(chunk)
    print(chunk)
    start = start + step

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
[5, 6, 7, 8, 9, 10, 11, 12, 13, 14]
[10, 11, 12, 13, 14, 15, 16, 17, 18, 19]
[15, 16, 17, 18, 19, 20, 21, 22, 23, 24]
[20, 21, 22, 23, 24, 25, 26, 27, 28, 29]
[25, 26, 27, 28, 29, 30, 31, 32, 33, 34]
[30, 31, 32, 33, 34, 35, 36, 37, 38, 39]
[35, 36, 37, 38, 39, 40, 41, 42, 43, 44]
[40, 41, 42, 43, 44, 45, 46, 47, 48, 49]
[45, 46, 47, 48, 49, 50, 51, 52, 53, 54]
[50, 51, 52, 53, 54, 55, 56, 57, 58, 59]
[55, 56, 57, 58, 59, 60, 61, 62, 63, 64]
[60, 61, 62, 63, 64, 65, 66, 67, 68, 69]
[65, 66, 67, 68, 69, 70, 71, 72, 73, 74]
[70, 71, 72, 73, 74, 75, 76, 77, 78, 79]
[75, 76, 77, 78, 79, 80, 81, 82, 83, 84]
[80, 81, 82, 83, 84, 85, 86, 87, 88, 89]
[85, 86, 87, 88, 89, 90, 91, 92, 93, 94]
[90, 91, 92, 93, 94, 95, 96, 97, 98, 99]


In [32]:
def sliding_window(text, size=1000, step=500):
    chunks = []
    start = 0
    text_length = len(text)

    while start < text_length:
        end = start + size
        chunk = text[start:end]
        chunks.append({'start': start, 'content': chunk})

        start = end - step

        if end >= text_length:
            break
            
    return chunks

In [33]:
len(sliding_window(results[0]['content'], size=3000, step=2500))

39

In [34]:
document_chunks = []

for doc in documents:
    if not doc.get('content'):
        continue
    copy = doc.copy()
    content = copy.pop('content')

    chunks = sliding_window(content, size=3000, step=1500)

    for i, chunk in enumerate(chunks):
        chunk.update(copy)
        chunk['chunk_id'] = i
        document_chunks.append(chunk)

In [35]:
document_chunks[10]

{'start': 9000,
 'content': 'cation=[BinaryClassification(\n        target="target",\n        prediction_labels="prediction")],\n    categorical_columns=["target", "prediction"])\n```\n\nAvailable options and defaults:\n\n```python\n    target: str = "target"\n    prediction_labels: Optional[str] = None\n    prediction_probas: Optional[str] = "prediction" #if probabilistic classification\n    pos_label: Label = 1 #name of the positive label\n    labels: Optional[Dict[Label, str]] = None\n```\n\n### Ranking\n\n#### RecSys\n\nTo evaluate recommender systems performance, you must map the columns with:\n\n- Prediction: this could be predicted score or rank.\n- Target: relevance labels (e.g., this could be an interaction result like user click or upvote, or a true relevance label)\n\nThe **target** column can contain either:\n\n- a binary label (where `1` is a positive outcome)\n- any scores (positive values, where a higher value corresponds to a better match or a more valuable user action)

In [36]:
chunk_index = Index(
    text_fields=["title", "description", "content"],
    keyword_fields=["filename"]
)
chunk_index.fit(document_chunks)

In [37]:
results = chunk_index.search(query)

In [38]:
from gitsource import chunk_documents

In [39]:
document_chunks = chunk_documents(documents, size=3000, step=1500)

## RAG

In [40]:
from openai import OpenAI

openai_client = OpenAI()

In [43]:
search_result = chunk_index.search(query, num_results=5)

In [45]:
query = 'how do I implement llm as a judge?'

In [46]:
import json

In [47]:
search_result_json = json.dumps(search_result, indent=2)

In [48]:
instructions = """
You're a course assistant, your task is to answer the QUESTION from the 
course students using the provided CONTEXT
"""

user_prompt = f"""
<QUESTION>
{query}
</QUESTION>

<CONTEXT>
{search_result_json}
</CONTEXT>
""".strip()

In [49]:
def llm(user_prompt, instructions=None, model='gpt-4o-mini'):

    messages = []

    if instructions is not None:
        messages.append({
            "role": "system",
            "content": instructions
        })

    messages.append({
        "role": "user",
        "content": user_prompt
    })

    response = openai_client.responses.create(
        model=model,
        input=messages
    )

    return response.output_text

In [50]:
answer = llm(user_prompt, instructions)

In [51]:
print(answer)

To implement an LLM (Large Language Model) as a judge, you can follow these key steps:

### 1. Setup Your Environment
- **Install the Evidently Library**:
  ```bash
  pip install evidently
  ```

- **Import Required Modules**:
  ```python
  import pandas as pd
  import numpy as np
  from evidently import Dataset, DataDefinition, Report, BinaryClassification
  from evidently.descriptors import *
  from evidently.presets import TextEvals, ValueStats, ClassificationPreset
  from evidently.metrics import *
  from evidently.llm.templates import BinaryClassificationPromptTemplate
  ```

- **Set Your OpenAI API Key**:
  ```python
  import os
  os.environ["OPENAI_API_KEY"] = "YOUR_KEY"
  ```

### 2. Create an Evaluation Dataset
- Make a dataset that contains:
  - **Questions**: User inputs.
  - **Target Responses**: Approved correct responses.
  - **New Responses**: Responses you want to evaluate.
  - **Manual Labels**: Labels indicating if the response is correct.

#### Example Code to Genera

In [52]:
def search(query):
    return chunk_index.search(query, num_results=5)

In [53]:
instructions = """
You're a course assistant, your task is to answer the QUESTION from the 
course students using the provided CONTEXT
"""

def build_prompt(query, search_results):
    search_result_json = json.dumps(search_result, indent=2)

    user_prompt = f"""
    <QUESTION>
    {query}
    </QUESTION>

    <CONTEXT>
    {search_result_json}
    </CONTEXT>
    """.strip()

    return user_prompt

In [54]:
def rag(query):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(prompt, instructions)
    return answer

In [55]:
rag('how do i implement llm as a judge?')

'To implement an LLM as a judge, follow these steps:\n\n1. **Installation and Setup**:\n   - Install the required library:\n     ```python\n     pip install evidently\n     ```\n   - Import necessary modules:\n     ```python\n     import pandas as pd\n     import numpy as np\n     from evidently import Dataset, Report\n     from evidently.llm.templates import BinaryClassificationPromptTemplate\n     ```\n   - Set your OpenAI API key:\n     ```python\n     import os\n     os.environ["OPENAI_API_KEY"] = "YOUR_KEY"\n     ```\n\n2. **Create an Evaluation Dataset**:\n   - Construct a toy dataset containing questions, approved responses, and new responses. Include manual labels to assess if the new responses are correct:\n     ```python\n     data = [\n         ["Hi there, how do I reset my password?", "To reset your password...", "To change your password...", "incorrect", "adds new information (contact support)"],\n         ...\n     ]\n     ```\n\n3. **Design the LLM Evaluator Prompt**:\n 